# 🏆 PSTU DataThon 2026 — V2: Leakage-Free Grand Master Solution

**Binary Classification | F1 Score @ 0.5 Threshold | No Calibration Tricks | Native Cat Handling**

---

## 🔴 V1 Post-Mortem: Why LB F1 (0.196) was 46% below CV F1 (0.362)

| Root Cause | V1 Issue | V2 Fix |
|------------|----------|--------|
| **Calibration shift leakage** | Affine shifts (+0.12 to +0.28) overfit OOF, don't transfer to test | **No calibration** — raw probabilities |
| **Target encoding leakage** | Smoothing=10 with 2333 cats memorizes rare labels | **Native cat handling** by CatBoost inside trees |
| **SMOTE over-aggression** | 0.5 ratio → 30k synthetic samples, poor generalization | **0.3 ratio** + scale_pos_weight |
| **KMeans on combined data** | Cluster distances peek at test distribution | **Removed** — only pure row-wise stats |
| **Weak regularization** | num_leaves=96, min_child=40 | **num_leaves=48, min_child=100, reg_alpha=0.1** |
| **Model diversity trap** | 3 different models + calibration destroys ensemble | **3-seed CatBoost ensemble** — same architecture, different initializations |

---

## 📋 V2 Strategy

- **CatBoost-only** (best V1 OOF: 0.373) × 3 seeds → rank-average ensemble
- **Native categorical features** — CatBoost handles feat_142/157/318/320/325/337 internally
- **6 row-wise features only** — mean, std, skew, kurt, iqr, zero_count
- **QuantileTransformer(output='normal')** for numerical features
- **No PCA** — CatBoost handles high dimensions well
- **No calibration shift** — raw probabilities, Kaggle thresholds at 0.5
- **SMOTE(0.3) + scale_pos_weight(~12)** — complementary imbalance handling
- **Stratified 5-Fold CV** — faster iteration, similar reliability


In [1]:
# ===================================================================
# CELL 1: Imports & Environment Setup
# ===================================================================
import numpy as np
import pandas as pd
import warnings, os, random, gc
warnings.filterwarnings('ignore')

from sklearn.preprocessing import QuantileTransformer, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from imblearn.over_sampling import SMOTE
from catboost import CatBoostClassifier, Pool
from scipy.stats import rankdata

# === Reproducibility ===
BASE_SEED = 42
np.random.seed(BASE_SEED)
random.seed(BASE_SEED)

# === Paths — auto-detect environment ===
KAGGLE_BASE = '/kaggle/input/competitions/pstu-data-thon-2026-vol-1'
if os.path.isdir(KAGGLE_BASE):
    TRAIN_PATH = f'{KAGGLE_BASE}/train.csv'
    TEST_PATH  = f'{KAGGLE_BASE}/test.csv'
    SAMPLE_PATH = f'{KAGGLE_BASE}/sample_submission.csv'
    print('Running on Kaggle')
elif os.path.isdir('./Dataset'):
    TRAIN_PATH = './Dataset/train.csv'
    TEST_PATH  = './Dataset/test.csv'
    SAMPLE_PATH = './Dataset/sample_submission.csv'
    print('Running locally')
else:
    TRAIN_PATH = '/kaggle/input/pstu-data-thon-2026-vol-1/train.csv'
    TEST_PATH  = '/kaggle/input/pstu-data-thon-2026-vol-1/test.csv'
    SAMPLE_PATH = '/kaggle/input/pstu-data-thon-2026-vol-1/sample_submission.csv'

print('All libraries loaded.')

Running on Kaggle
All libraries loaded.


In [2]:
# ===================================================================
# CELL 2: Configuration — Leakage-Free, Conservative Pipeline
# ===================================================================

CFG = {
    'seed': 42,
    'ensemble_seeds': [42, 123, 456],  # 3 seeds for CatBoost ensemble
    'n_folds': 5,                       # Stratified 5-Fold CV
    
    # --- SMOTE ---
    'smote_strategy': 0.3,              # Gentler: ~3.3:1 ratio after SMOTE
    'scale_pos_weight': 12.0,           # Complementary to SMOTE (~73k/3k/2)
    
    # --- Feature Engineering (conservative) ---
    'use_row_stats': True,              # Only 6 row-wise stats
    
    # --- CatBoost (primary model) ---
    'cb_params': {
        'loss_function': 'Logloss',
        'eval_metric': 'F1',            # Optimize F1 directly
        'iterations': 5000,
        'learning_rate': 0.015,
        'depth': 5,                     # Shallower trees — less overfit
        'l2_leaf_reg': 5.0,             # Stronger L2 regularization
        'random_strength': 1.5,         # More randomness for diversity
        'bagging_temperature': 0.8,
        'border_count': 254,
        'grow_policy': 'SymmetricTree',
        'min_data_in_leaf': 50,         # Larger leaves — less overfit
        'one_hot_max_size': 10,         # One-hot encode small cats
        'od_type': 'Iter',
        'od_wait': 150,
        'thread_count': -1,
        'verbose': 0,
        'allow_writing_files': False,
        'auto_class_weights': 'Balanced',  # Auto class weighting
    },
}

print(f'V2 Config: {CFG["n_folds"]}-Fold | SMOTE={CFG["smote_strategy"]} | scale_pos_weight={CFG["scale_pos_weight"]}')
print(f'Ensemble: {len(CFG["ensemble_seeds"])} CatBoost seeds | depth={CFG["cb_params"]["depth"]} | l2_reg={CFG["cb_params"]["l2_leaf_reg"]}')

V2 Config: 5-Fold | SMOTE=0.3 | scale_pos_weight=12.0
Ensemble: 3 CatBoost seeds | depth=5 | l2_reg=5.0


In [3]:
# ===================================================================
# CELL 3: Data Loading & Column Identification
# ===================================================================

print('Loading datasets...')
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')

# Extract target and IDs
TARGET_COL = 'TARGET'
y = train[TARGET_COL].copy()

if 'id' in test.columns:
    test_ids = test['id'].copy()
    X_test_raw = test.drop(columns=['id'])
else:
    test_ids = pd.Series(range(len(test)), name='id')
    X_test_raw = test.copy()

X_train_raw = train.drop(columns=[TARGET_COL])

# --- Identify column types ---
feat_cols = [c for c in X_train_raw.columns if c.startswith('feat_')]
cat_cols = X_train_raw[feat_cols].select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in feat_cols if c not in cat_cols]

# --- Label-encode categoricals for numeric pipeline (CatBoost gets originals) ---
cat_encoders = {}
X_train_cat_encoded = pd.DataFrame(index=X_train_raw.index)
X_test_cat_encoded  = pd.DataFrame(index=X_test_raw.index)

for col in cat_cols:
    le = LabelEncoder()
    all_vals = pd.concat([X_train_raw[col], X_test_raw[col]]).astype(str)
    le.fit(all_vals)
    X_train_cat_encoded[col] = le.transform(X_train_raw[col].astype(str))
    X_test_cat_encoded[col]  = le.transform(X_test_raw[col].astype(str))
    cat_encoders[col] = le

print(f'Features: {len(num_cols)} numerical + {len(cat_cols)} categorical')
print(f'Target distribution: 0={ (y==0).sum():,} ({(y==0).mean()*100:.2f}%) | 1={(y==1).sum():,} ({(y==1).mean()*100:.2f}%)')

Loading datasets...
Train: (76020, 351)
Test:  (60654, 351)
Features: 344 numerical + 6 categorical
Target distribution: 0=73,012 (96.04%) | 1=3,008 (3.96%)


In [4]:
# ===================================================================
# CELL 4: Feature Cleaning + Conservative Engineering
# ===================================================================

# --- 4a. Ensure numeric ---
X_num_tr = X_train_raw[num_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)
X_num_te = X_test_raw[num_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)

# --- 4b. Drop zero-variance ---
variances = X_num_tr.var()
zero_var = variances[variances <= 1e-12].index.tolist()
print(f'Dropping {len(zero_var)} zero-variance features')

# --- 4c. Drop duplicates ---
arr_tr = X_num_tr.values.astype(np.float64)
dup_drop = set()
sigs = {}
for i, c in enumerate(num_cols):
    if c in zero_var: continue
    col = arr_tr[:, i]
    sig = (hash(col[:500].tobytes()), hash(col[500:1000].tobytes()), int(col.var()*1e6))
    if sig in sigs:
        j = sigs[sig]
        if np.array_equal(col, arr_tr[:, j]):
            dup_drop.add(c)
    else:
        sigs[sig] = i
print(f'Dropping {len(dup_drop)} duplicate features')

all_drop = set(zero_var) | dup_drop
keep_num = [c for c in num_cols if c not in all_drop]

X_num_tr = X_num_tr[keep_num]
X_num_te = X_num_te[keep_num]
print(f'Kept {len(keep_num)} numerical features')

# --- 4d. Conservative row-wise features (no leakage path) ---
if CFG['use_row_stats']:
    print('Adding row-wise statistical features...')
    arr_tr_np = X_num_tr.values.astype(np.float64)
    arr_te_np = X_num_te.values.astype(np.float64)
    
    row_feats_tr = {}
    row_feats_te = {}
    
    row_feats_tr['row_mean'] = arr_tr_np.mean(axis=1)
    row_feats_tr['row_std']  = arr_tr_np.std(axis=1)
    row_feats_tr['row_iqr']  = np.percentile(arr_tr_np, 75, axis=1) - np.percentile(arr_tr_np, 25, axis=1)
    row_feats_tr['row_zero'] = (arr_tr_np == 0).sum(axis=1)
    from scipy.stats import skew, kurtosis
    row_feats_tr['row_skew'] = skew(arr_tr_np, axis=1)
    row_feats_tr['row_kurt'] = kurtosis(arr_tr_np, axis=1)
    
    row_feats_te['row_mean'] = arr_te_np.mean(axis=1)
    row_feats_te['row_std']  = arr_te_np.std(axis=1)
    row_feats_te['row_iqr']  = np.percentile(arr_te_np, 75, axis=1) - np.percentile(arr_te_np, 25, axis=1)
    row_feats_te['row_zero'] = (arr_te_np == 0).sum(axis=1)
    row_feats_te['row_skew'] = skew(arr_te_np, axis=1)
    row_feats_te['row_kurt'] = kurtosis(arr_te_np, axis=1)
    
    df_row_tr = pd.DataFrame(row_feats_tr, index=X_num_tr.index)
    df_row_te = pd.DataFrame(row_feats_te, index=X_num_te.index)
    print(f'  Created {df_row_tr.shape[1]} row-wise features')

# --- 4e. Combine: numeric + label-encoded categoricals + row stats ---
# Label-encoded cats are integers -> safe for SMOTE + QuantileTransformer
# CatBoost will treat them as categorical via cat_features indices
X_tr_all_numeric = pd.concat([
    X_num_tr.reset_index(drop=True),
    X_train_cat_encoded.reset_index(drop=True),
], axis=1)

X_te_all_numeric = pd.concat([
    X_num_te.reset_index(drop=True),
    X_test_cat_encoded.reset_index(drop=True),
], axis=1)

if CFG['use_row_stats']:
    X_tr_all_numeric = pd.concat([X_tr_all_numeric, df_row_tr.reset_index(drop=True)], axis=1)
    X_te_all_numeric = pd.concat([X_te_all_numeric, df_row_te.reset_index(drop=True)], axis=1)

# Track where categorical columns land in the combined numeric matrix
# They are right after the numerical columns
cat_start_idx = len(keep_num)
cat_indices = list(range(cat_start_idx, cat_start_idx + len(cat_cols)))

print(f'Final all-numeric matrix: {X_tr_all_numeric.shape[1]} features')
print(f'  Numerical:           {len(keep_num)}')
print(f'  Label-encoded cats:  {len(cat_cols)} (indices {cat_indices[0]}-{cat_indices[-1]})')
if CFG['use_row_stats']:
    print(f'  Row-wise stats:      {df_row_tr.shape[1]}')
print(f'  CatBoost cat_indices: {cat_indices}')

del X_train_raw, X_test_raw, X_num_tr, X_num_te, arr_tr, arr_tr_np, arr_te_np
gc.collect()

Dropping 28 zero-variance features
Dropping 15 duplicate features
Kept 301 numerical features
Adding row-wise statistical features...
  Created 6 row-wise features
Final all-numeric matrix: 313 features
  Numerical:           301
  Label-encoded cats:  6 (indices 301-306)
  Row-wise stats:      6
  CatBoost cat_indices: [301, 302, 303, 304, 305, 306]


0

In [5]:
# ===================================================================
# CELL 5: QuantileTransform (numerical only) + Combine with Raw Cats
# ===================================================================
# CRITICAL: QT is applied ONLY to truly numerical columns.
# Label-encoded categorical columns are kept as raw integers so
# CatBoost can split on them properly via cat_features.
# SMOTE also works because everything is numeric (ints + floats).

X_tr_all_numeric = X_tr_all_numeric.fillna(0).replace([np.inf, -np.inf], 0).astype(np.float32)
X_te_all_numeric = X_te_all_numeric.fillna(0).replace([np.inf, -np.inf], 0).astype(np.float32)

# Identify which columns are truly numerical (not label-encoded cats)
num_feature_indices = [i for i in range(X_tr_all_numeric.shape[1]) if i not in cat_indices]

print(f'Applying QuantileTransformer to {len(num_feature_indices)} numerical columns...')
print(f'  (skipping {len(cat_indices)} label-encoded categorical columns)')

# QT only on numerical columns
X_tr_num_part = X_tr_all_numeric.iloc[:, num_feature_indices].values
X_te_num_part = X_te_all_numeric.iloc[:, num_feature_indices].values
X_tr_cat_part = X_tr_all_numeric.iloc[:, cat_indices].values.astype(np.int32)
X_te_cat_part = X_te_all_numeric.iloc[:, cat_indices].values.astype(np.int32)

qt = QuantileTransformer(
    n_quantiles=min(2000, len(X_tr_num_part)),
    output_distribution='normal',
    random_state=CFG['seed'],
    subsample=200_000
)
X_tr_qt = qt.fit_transform(X_tr_num_part).astype(np.float32)
X_te_qt = qt.transform(X_te_num_part).astype(np.float32)

print(f'  QT numerical: {X_tr_qt.shape}')

# Combine: QT-transformed numerical + raw label-encoded categoricals
# Keep cat columns as int32 — CatBoost requires non-float dtype for cat_features
# hstack(float32, int32) → float64 (safe for SMOTE; wrap in DataFrame for CatBoost)
X_tr_final = np.hstack([X_tr_qt, X_tr_cat_part])
X_te_final = np.hstack([X_te_qt, X_te_cat_part])

# Update cat_indices to reflect positions in final combined array
# Cats are at the end: [0...num_qt-1, num_qt...num_qt+n_cats-1]
num_qt_cols = X_tr_qt.shape[1]
cat_indices_final = list(range(num_qt_cols, num_qt_cols + len(cat_cols)))

print(f'Final matrix: Train {X_tr_final.shape}, Test {X_te_final.shape}')
print(f'  QT numerical columns: {num_qt_cols}')
print(f'  Raw cat columns:      {len(cat_cols)} (indices {cat_indices_final[0]}-{cat_indices_final[-1]})')

del X_tr_all_numeric, X_te_all_numeric, X_tr_num_part, X_te_num_part
del X_tr_cat_part, X_te_cat_part, X_tr_qt, X_te_qt
gc.collect()

Applying QuantileTransformer to 307 numerical columns...
  (skipping 6 label-encoded categorical columns)
  QT numerical: (76020, 307)
Final matrix: Train (76020, 313), Test (60654, 313)
  QT numerical columns: 307
  Raw cat columns:      6 (indices 307-312)


0

In [6]:
# ===================================================================
# CELL 6: 5-Fold CV — CatBoost × 3 Seeds → Rank-Average Ensemble
# ===================================================================
from sklearn.model_selection import StratifiedKFold

ENSEMBLE_SEEDS = CFG['ensemble_seeds']
N_FOLDS = CFG['n_folds']

# Helper: convert numpy array to DataFrame with cat columns as strings
# (strings avoid CatBoost's dtype.kind=='f' rejection and float-value issues)
def make_cb_df(arr, cat_indices):
    """Wrap numpy array in DataFrame with categorical columns as strings."""
    df = pd.DataFrame(arr)
    for ci in cat_indices:
        # SMOTE interpolates cat values (e.g. 469+471→470.3); round→int→str
        df.iloc[:, ci] = df.iloc[:, ci].round().astype(int).astype(str)
    return df

# Pre-convert test data once (cat cols are already clean ints from label-encode)
X_te_cb_df = make_cb_df(X_te_final, cat_indices_final)

# OOF predictions per seed
all_oof = {}   # {seed: oof_array}
all_test = {}  # {seed: test_array}
all_fold_scores = {}  # {seed: [fold_f1, ...]}

for seed_idx, seed in enumerate(ENSEMBLE_SEEDS):
    print(f'\n{"="*60}')
    print(f'  SEED {seed_idx+1}/{len(ENSEMBLE_SEEDS)} (seed={seed})')
    print(f'{"="*60}')
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof_preds = np.zeros(len(X_tr_final), dtype=np.float32)
    test_preds = np.zeros(len(X_te_final), dtype=np.float32)
    fold_scores = []
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_tr_final, y)):
        X_tr_fold = X_tr_final[tr_idx]
        X_val_fold = X_tr_final[val_idx]
        y_tr_fold = y.iloc[tr_idx].values
        y_val_fold = y.iloc[val_idx].values
        
        # --- SMOTE (gentle ratio) — all columns are numeric (float64 from hstack) ---
        sm = SMOTE(sampling_strategy=CFG['smote_strategy'], random_state=seed+fold)
        X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_fold, y_tr_fold)
        ratio = (y_tr_sm == 0).sum() / (y_tr_sm == 1).sum()
        print(f'  Fold {fold+1}: SMOTE {y_tr_fold.sum()}->{y_tr_sm.sum()} pos (ratio={ratio:.1f}:1) | ', end='')
        
        # --- Wrap in DataFrame for CatBoost (cat columns → string to avoid dtype issues) ---
        X_tr_sm_cb = make_cb_df(X_tr_sm, cat_indices_final)
        X_val_cb   = make_cb_df(X_val_fold, cat_indices_final)
        
        # --- CatBoost with native categorical handling ---
        cb_params = CFG['cb_params'].copy()
        cb_params['random_seed'] = seed
        
        cb = CatBoostClassifier(**cb_params)
        cb.fit(
            X_tr_sm_cb, y_tr_sm,
            cat_features=cat_indices_final,
            eval_set=[(X_val_cb, y_val_fold)],
            early_stopping_rounds=150,
            verbose=0,
        )
        
        oof_preds[val_idx] = cb.predict_proba(X_val_cb)[:, 1]
        test_preds += cb.predict_proba(X_te_cb_df)[:, 1] / N_FOLDS
        
        f1_val = f1_score(y_val_fold, (oof_preds[val_idx] >= 0.5).astype(int))
        fold_scores.append(f1_val)
        print(f'F1={f1_val:.5f}')
        
        del X_tr_fold, X_val_fold, y_tr_fold, y_val_fold, X_tr_sm, X_tr_sm_cb, X_val_cb, cb
        gc.collect()
    
    all_oof[seed] = oof_preds
    all_test[seed] = test_preds
    all_fold_scores[seed] = fold_scores
    
    oof_f1 = f1_score(y, (oof_preds >= 0.5).astype(int))
    print(f'  Seed {seed} OOF F1 @ 0.5: {oof_f1:.5f} | Fold F1: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}')

# --- Ensemble: rank-average across seeds ---
print(f'\n{"="*60}')
print(f'  ENSEMBLE: Rank-Average of {len(ENSEMBLE_SEEDS)} CatBoost Seeds')
print(f'{"="*60}')

# Compute ranks per seed
rank_oofs = {}
rank_tests = {}
for seed in ENSEMBLE_SEEDS:
    rank_oofs[seed] = rankdata(all_oof[seed]) / len(all_oof[seed])
    rank_tests[seed] = rankdata(all_test[seed]) / len(all_test[seed])

# Average ranks
oof_rank_avg = np.mean([rank_oofs[s] for s in ENSEMBLE_SEEDS], axis=0)
test_rank_avg = np.mean([rank_tests[s] for s in ENSEMBLE_SEEDS], axis=0)

# Also try weighted probability blend for comparison
oof_prob_avg = np.mean([all_oof[s] for s in ENSEMBLE_SEEDS], axis=0)
test_prob_avg = np.mean([all_test[s] for s in ENSEMBLE_SEEDS], axis=0)

f1_rank = f1_score(y, (oof_rank_avg >= 0.5).astype(int))
f1_prob = f1_score(y, (oof_prob_avg >= 0.5).astype(int))

print(f'  Rank-average OOF F1:    {f1_rank:.5f}')
print(f'  Probability-avg OOF F1: {f1_prob:.5f}')

# Pick best method
if f1_rank >= f1_prob:
    print(f'  >>> Using RANK-AVERAGE ensemble')
    USE_RANK = True
else:
    print(f'  >>> Using PROBABILITY-AVERAGE ensemble')
    USE_RANK = False

for seed in ENSEMBLE_SEEDS:
    print(f'  Seed {seed}: Fold F1 = {np.mean(all_fold_scores[seed]):.4f} +/- {np.std(all_fold_scores[seed]):.4f}')


  SEED 1/3 (seed=42)
  Fold 1: SMOTE 2407->17522 pos (ratio=3.3:1) | F1=0.27091
  Fold 2: SMOTE 2407->17522 pos (ratio=3.3:1) | F1=0.24889
  Fold 3: SMOTE 2406->17523 pos (ratio=3.3:1) | F1=0.25092
  Fold 4: SMOTE 2406->17523 pos (ratio=3.3:1) | F1=0.34738
  Fold 5: SMOTE 2406->17523 pos (ratio=3.3:1) | F1=0.31698
  Seed 42 OOF F1 @ 0.5: 0.27829 | Fold F1: 0.2870 +/- 0.0389

  SEED 2/3 (seed=123)
  Fold 1: SMOTE 2407->17522 pos (ratio=3.3:1) | F1=0.29222
  Fold 2: SMOTE 2407->17522 pos (ratio=3.3:1) | F1=0.23543
  Fold 3: SMOTE 2406->17523 pos (ratio=3.3:1) | F1=0.23975
  Fold 4: SMOTE 2406->17523 pos (ratio=3.3:1) | F1=0.35730
  Fold 5: SMOTE 2406->17523 pos (ratio=3.3:1) | F1=0.28082
  Seed 123 OOF F1 @ 0.5: 0.27092 | Fold F1: 0.2811 +/- 0.0441

  SEED 3/3 (seed=456)
  Fold 1: SMOTE 2407->17522 pos (ratio=3.3:1) | F1=0.24039
  Fold 2: SMOTE 2407->17522 pos (ratio=3.3:1) | F1=0.33112
  Fold 3: SMOTE 2406->17523 pos (ratio=3.3:1) | F1=0.26280
  Fold 4: SMOTE 2406->17523 pos (ratio=3.3

In [7]:
# ===================================================================
# CELL 7: Generate Submission — Direct Probabilities, No Calibration
# ===================================================================

print('Generating submission...')

if USE_RANK:
    final_preds = test_rank_avg
    print('  Using rank-average predictions')
else:
    final_preds = test_prob_avg
    print('  Using probability-average predictions')

# NO calibration shift. Submit raw probabilities.
# Kaggle evaluates F1 at exactly 0.5 threshold.
# The model learns naturally calibrated probabilities via LogLoss.

# --- Build submission ---
submission = pd.DataFrame({
    'id': test_ids.values,
    'TARGET': final_preds
})

# --- Statistics ---
print(f'\n  Prediction stats:')
print(f'    Mean:  {final_preds.mean():.4f}')
print(f'    Median: {np.median(final_preds):.4f}')
print(f'    Std:   {final_preds.std():.4f}')
print(f'    Min/Max: {final_preds.min():.4f} / {final_preds.max():.4f}')
print(f'    >=0.5: {(final_preds >= 0.5).sum():,} ({(final_preds >= 0.5).mean()*100:.2f}%)')

# Also create binary version for platforms that need it
submission_binary = pd.DataFrame({
    'id': test_ids.values,
    'TARGET': (final_preds >= 0.5).astype(int)
})

# --- Save both ---
submission.to_csv('submission.csv', index=False)
submission_binary.to_csv('submission_binary.csv', index=False)

print(f'\n  Saved: submission.csv (probabilities, {len(submission):,} rows)')
print(f'  Saved: submission_binary.csv (0/1 binary, {len(submission_binary):,} rows)')
print(f'\n  Preview (first 10):')
print(submission.head(10).to_string(index=False))

# Show some TARGET=1 rows
ones = submission_binary[submission_binary['TARGET'] == 1]
if len(ones) > 0:
    print(f'\n  Sample rows with TARGET=1 ({len(ones)} total):')
    print(ones.head(8).to_string(index=False))

Generating submission...
  Using probability-average predictions

  Prediction stats:
    Mean:  0.2413
    Median: 0.2025
    Std:   0.1337
    Min/Max: 0.0832 / 0.8458
    >=0.5: 4,276 (7.05%)

  Saved: submission.csv (probabilities, 60,654 rows)
  Saved: submission_binary.csv (0/1 binary, 60,654 rows)

  Preview (first 10):
   id   TARGET
 3496 0.156564
17271 0.239602
44259 0.139516
64996 0.284508
23333 0.105360
56223 0.118556
19215 0.122370
58525 0.407816
53631 0.190279
53855 0.268205

  Sample rows with TARGET=1 (4276 total):
   id  TARGET
19565       1
44135       1
35293       1
36677       1
11020       1
 8685       1
32141       1
 2315       1


In [8]:
# ===================================================================
# CELL 8: Performance Summary
# ===================================================================

print('='*60)
print('  V2 GRAND MASTER SOLUTION — PERFORMANCE SUMMARY')
print('='*60)
print(f'  Model:            CatBoost (depth={CFG["cb_params"]["depth"]}, l2_reg={CFG["cb_params"]["l2_leaf_reg"]})')
print(f'  Ensemble:         {len(ENSEMBLE_SEEDS)} seeds x rank-average')
print(f'  CV:               {N_FOLDS}-Fold Stratified')
print(f'  Imbalance:        SMOTE({CFG["smote_strategy"]}) + auto_class_weights')
print(f'  Categoricals:     Native CatBoost handling (label-encoded, no TE leakage)')
print(f'  Features:         {X_tr_final.shape[1]} (QT numerical + {len(cat_cols)} raw cat + row stats)')
print(f'  Calibration:      NONE — raw probabilities via LogLoss')
print('-'*60)
print(f'  OOF F1 @ 0.5:')
for seed in ENSEMBLE_SEEDS:
    seed_f1 = f1_score(y, (all_oof[seed] >= 0.5).astype(int))
    print(f'    CatBoost seed={seed}:  {seed_f1:.5f}')
print(f'    Ensemble (rank-avg):    {f1_rank:.5f}')
print(f'    Ensemble (prob-avg):    {f1_prob:.5f}')
print('-'*60)
print(f'  Fold F1 (mean +/- std):')
for seed in ENSEMBLE_SEEDS:
    print(f'    Seed {seed}:  {np.mean(all_fold_scores[seed]):.4f} +/- {np.std(all_fold_scores[seed]):.4f}')
print('='*60)
print(f'  Test predicted positive rate: {(final_preds >= 0.5).mean()*100:.2f}% (train: {(y==1).mean()*100:.2f}%)')
print(f'')
print(f'  V1 LB F1: 0.1957  |  V1 CV F1: 0.3624  |  CV-LB Gap: 46%')
print(f'  V2 aims to close this gap via:')
print(f'    1. No calibration shift (removes +0.12-0.28 distortion)')
print(f'    2. Native cat handling via CatBoost (removes TE leakage)')
print(f'    3. Gentler SMOTE (0.3 vs 0.5)')
print(f'    4. Stronger regularization (depth=5, l2_reg=5.0)')
print(f'    5. Simpler features (no KMeans clusters)')
print(f'')
print(f'  Outputs: submission.csv | submission_binary.csv')

  V2 GRAND MASTER SOLUTION — PERFORMANCE SUMMARY
  Model:            CatBoost (depth=5, l2_reg=5.0)
  Ensemble:         3 seeds x rank-average
  CV:               5-Fold Stratified
  Imbalance:        SMOTE(0.3) + auto_class_weights
  Categoricals:     Native CatBoost handling (label-encoded, no TE leakage)
  Features:         313 (QT numerical + 6 raw cat + row stats)
  Calibration:      NONE — raw probabilities via LogLoss
------------------------------------------------------------
  OOF F1 @ 0.5:
    CatBoost seed=42:  0.27829
    CatBoost seed=123:  0.27092
    CatBoost seed=456:  0.26833
    Ensemble (rank-avg):    0.13809
    Ensemble (prob-avg):    0.28724
------------------------------------------------------------
  Fold F1 (mean +/- std):
    Seed 42:  0.2870 +/- 0.0389
    Seed 123:  0.2811 +/- 0.0441
    Seed 456:  0.2755 +/- 0.0338
  Test predicted positive rate: 7.05% (train: 3.96%)

  V1 LB F1: 0.1957  |  V1 CV F1: 0.3624  |  CV-LB Gap: 46%
  V2 aims to close this gap v